# Assignment 4 — Retrieval-Augmented Generation

**Assessment:** 13 points, all autograded.

All five tasks are tested **offline** with inputs beyond the ones shown. Task 5 also provides a
small evaluation harness that calls your functions from Tasks 1, 3 and 4 — plan about thirty
minutes for the whole assignment.

Each answer cell contains `raise NotImplementedError()` — replace it with your code and keep the function names and signatures unchanged.

## Setup — run first, do not modify

In [1]:
import numpy as np

TEXT = "abcdefghij"          # 10 characters, easy to check by hand

RETRIEVED = ["d3", "d7", "d1", "d9", "d2"]
RELEVANT  = {"d1", "d2"}

QUERIES = [
    (["a", "b", "c"], {"b"}),      # first relevant at rank 2 -> 1/2
    (["x", "y", "z"], {"q"}),      # none found               -> 0
    (["m", "n"],      {"m", "n"}), # first relevant at rank 1 -> 1
]
print("setup complete")

setup complete


## Task 1 — chunking (3 points)

Write `chunk_text(text, size, overlap)` returning a list of chunks such that

* each chunk is at most `size` characters long and none is empty,
* consecutive chunks start `size - overlap` characters apart,
* the text is covered to its end: stop as soon as a chunk reaches the last character — no
  shorter chunks follow it,
* leading and trailing whitespace of the whole text is stripped first,
* an empty or whitespace-only text gives `[]`,
* `size <= 0`, `overlap < 0` or `overlap >= size` raises `ValueError`.

For `chunk_text("abcdefghij", 4, 1)` the step is 3, so the result is
`["abcd", "defg", "ghij"]` — not `["abcd", "defg", "ghij", "j"]`, because `"ghij"` already reaches the end.

In [2]:
def chunk_text( text, size, overlap ):
    if size <= 0 or overlap < 0 or overlap >= size:
        raise ValueError( "Error" )
    text = text.strip()
    if not text:
        return []
    chunks, start, step = [], 0, size - overlap
    while start < len( text ):
        end = min( start + size, len( text ) )
        chunks.append( text[start:end] )
        if end == len( text ):
            break
        start += step
    return chunks

In [3]:
c = chunk_text(TEXT, 4, 1)
assert isinstance(c, list) and all(isinstance(x, str) for x in c)

## Task 2 — fitting the prompt budget (2 points)

Write `select_context(chunks, max_chars)` returning the **longest prefix** of `chunks` whose
total length fits in `max_chars`. The chunks will be joined with a blank line (`"\n\n"`), so
count two extra characters for every separator between consecutive chunks: `["aaa", "bbb"]`
needs 3 + 2 + 3 = 8 characters. Stop at the first chunk that does not fit; do not skip it and
continue. An empty list gives `[]`; a negative `max_chars` raises `ValueError`.

In [4]:
def select_context( chunks, max_chars ):
    if max_chars < 0:
        raise ValueError( "Error" )
    selected = []
    total_length = 0
    for chunk in chunks:
        separator_length = 2 if selected else 0
        required_length = separator_length + len( chunk )
        if total_length + required_length > max_chars:
            break
        selected.append( chunk )
        total_length += required_length
    return selected

In [5]:
s = select_context(['aaa', 'bbb'], 100)
assert isinstance(s, list)

## Task 3 — hit@k and recall@k (2 points)

Write `hit_at_k(retrieved, relevant, k)` returning `1.0` if at least one of the first `k`
retrieved identifiers is relevant, else `0.0`.

Write `recall_at_k(retrieved, relevant, k)` returning the fraction of the **distinct** relevant
identifiers that appear among the first `k` retrieved.

Both return `0.0` when `relevant` is empty or `k == 0`, and raise `ValueError` for `k < 0`.

In [6]:
def hit_at_k( retrieved, relevant, k ):
    if k < 0:
        raise ValueError( "Error" )
    relevant = set( relevant )
    if k == 0 or not relevant:
        return 0.0
    retrieved_at_k = set( retrieved[:k] )
    found = retrieved_at_k & relevant
    return 1.0 if found else 0.0


def recall_at_k( retrieved, relevant, k ):
    if k < 0:
        raise ValueError( "Error" )
    relevant = set( relevant )
    if k == 0 or not relevant:
        return 0.0
    retrieved_at_k = set( retrieved[:k] )
    found = retrieved_at_k & relevant
    return len( found ) / len( relevant )

In [7]:
v = recall_at_k(RETRIEVED, RELEVANT, 3)
assert 0.0 <= float(v) <= 1.0

## Task 4 — mean reciprocal rank with a cutoff (3 points)

Write `reciprocal_rank(retrieved, relevant, k=None)` returning `1/rank` of the **first**
relevant identifier among the first `k` retrieved (all of them when `k` is `None`), counting
ranks from one, or `0.0` if none is present.

Write `mean_reciprocal_rank(queries, k=None)` where `queries` is a sequence of
`(retrieved, relevant)` pairs, returning the mean of the reciprocal ranks at cutoff `k`. An
empty input gives `0.0`. A cutoff `k = 0` gives `0.0`; negative `k` is not tested.

In [8]:
def reciprocal_rank( retrieved, relevant, k=None ):
    if k is None:
        selected = retrieved
    else:
        selected = retrieved[:k]
    rank = 1
    for identifier in selected:
        if identifier in relevant:
            return 1.0 / rank
        rank += 1
    return 0.0

def mean_reciprocal_rank( queries, k=None ):
    if not queries:
        return 0.0
    total = 0.0
    for retrieved, relevant in queries:
        total += reciprocal_rank( retrieved, relevant, k )
    return total / len( queries )

In [9]:
v = mean_reciprocal_rank(QUERIES)
assert 0.0 <= float(v) <= 1.0

## Task 5 — Compare retrieval configurations (3 points)

Implement `best_configurations(metrics)`. The input is a non-empty dictionary mapping
configuration names (strings) to dictionaries with numeric `hit@3` and `MRR@10` values
in `[0, 1]`. Select configurations with the highest `hit@3`; among those, select the
highest `MRR@10`. Return **all** configurations tied on both metrics as an alphabetically
sorted list of names. Compare values exactly as supplied; do not round or mutate the input.

The offline harness below evaluates two chunking configurations and a TF-IDF retriever. It calls
your `chunk_text`, `hit_at_k` and `reciprocal_rank`, so run it only after Tasks 1, 3 and 4 pass.
After implementing the function, try `best_configurations(results.to_dict(orient="index"))`.
Tests also use other names, values and numbers of configurations; do not hard-code the results.
(The harness scores a hash-based and a TF-IDF retriever, not real embeddings — the numbers show the
method, not the quality of any embedding model.)

In [10]:
import hashlib, pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

DOCUMENTS = {'organisation.md': 'This fictional study centre provides practical workshops and self-study '
                    'resources for data analysis. Participants work with examples that can be '
                    'inspected and rerun in a local Python environment. Workshops use a computer '
                    'laboratory with shared demonstration screens and individual workstations. The '
                    'advanced workshop covers neural networks, image classification and transfer '
                    'learning. Questions about methods are discussed with a workshop tutor, while '
                    'questions about access to equipment are handled by the support desk. The '
                    'examples in this handbook describe a demonstration setting, not the '
                    'organisation of an actual course.',
 'assessment.md': 'The study centre uses formative assessment to identify concepts that need '
                  'further practice. Feedback distinguishes conceptual understanding, '
                  'implementation quality and interpretation of results. Practice notebooks '
                  'combine executable examples with short written explanations. Automated checks '
                  'verify specified function behaviour on several inputs, but a successful check '
                  'does not establish that every possible input has been handled. Written '
                  'explanations are reviewed by a tutor. Participants retain responsibility for '
                  'checking any generated code and explaining how it works. The assessment '
                  'criteria are correctness, reproducibility and clarity.',
 'project.md': 'The demonstration project investigates a classification task using a labelled '
               'dataset. The target variable has to be nominal. The report describes the research '
               'question, the dataset and the intended prediction setting before introducing a '
               'baseline model. The report includes a methods section, an evaluation table and a '
               'discussion of limitations. Model selection uses training data, while a separate '
               'test set supports the final comparison. A peer review examines reproducibility and '
               'the interpretation of results. Reviewers state their evidence and distinguish '
               'observed defects from suggestions for further work. The final presentation '
               'explains the main findings and their practical limitations.',
 'software.md': 'Examples are provided as Jupyter notebooks containing both explanation and '
                'executable code. A read-only resource folder holds the example notebooks and '
                'input files. The runtime environment includes NumPy, pandas, matplotlib and '
                'scikit-learn. Package versions are recorded alongside the examples so that '
                'another reader can reproduce the computation. A fresh kernel run checks whether '
                'cells execute in their displayed order without relying on earlier interactive '
                'state. Optional language-model requests use a configured endpoint. Credentials '
                'are read from environment variables and are not written into notebook cells.'}
GOLD = [('What does the project report include?',
  'project.md',
  'The report includes a methods section, an evaluation table and a discussion of limitations'),
 ('What does the advanced workshop cover?',
  'organisation.md',
  'The advanced workshop covers neural networks, image classification and transfer learning'),
 ('What are the assessment criteria?',
  'assessment.md',
  'The assessment criteria are correctness, reproducibility and clarity'),
 ('Where are the example notebooks stored?',
  'software.md',
  'A read-only resource folder holds the example notebooks and input files'),
 ('Must the target variable be nominal?', 'project.md', 'The target variable has to be nominal'),
 ('Who reviews written explanations?',
  'assessment.md',
  'Written explanations are reviewed by a tutor'),
 ('Where do the workshops take place?',
  'organisation.md',
  'Workshops use a computer laboratory with shared demonstration screens and individual '
  'workstations'),
 ('What does a peer review examine?',
  'project.md',
  'A peer review examines reproducibility and the interpretation of results')]

def embed_hash(texts, dim=256):
    out = np.zeros((len(texts), dim), dtype=np.float32)
    for i, text in enumerate(texts):
        s = " " + text.lower().strip() + " "
        for j in range(len(s) - 2):
            out[i, int(hashlib.md5(s[j:j+3].encode()).hexdigest(), 16) % dim] += 1.0
    return out

def l2n(M):
    M = np.asarray(M, dtype=np.float32); return M / np.maximum(np.linalg.norm(M, axis=1, keepdims=True), 1e-12)

def build(size, overlap):
    chunks = []
    for name, body in DOCUMENTS.items():
        for i, piece in enumerate(chunk_text(body, size, overlap)):
            start = body.find(piece); chunks.append({"id": f"{name}#{i}", "doc": name, "start": start, "end": start + len(piece), "text": piece})
    return chunks

def relevant_ids(doc, answer, chunks):
    s = DOCUMENTS[doc].find(answer)
    assert s >= 0, f"answer text not found in {doc}: {answer!r}"
    e = s + len(answer)
    return {c["id"] for c in chunks if c["doc"] == doc and c["start"] < e and c["end"] > s}

def rank_hash(chunks, question, depth=10):
    ix = l2n(embed_hash([c["text"] for c in chunks])); q = l2n(embed_hash([question]))[0]
    return [chunks[i]["id"] for i in np.argsort(-(ix @ q))[:depth]]

def rank_tfidf(chunks, question, depth=10):
    vec = TfidfVectorizer(ngram_range=(1, 2)).fit([c["text"] for c in chunks])
    ix = vec.transform([c["text"] for c in chunks]); q = vec.transform([question])
    return [chunks[i]["id"] for i in np.argsort(-(ix @ q.T).toarray().ravel())[:depth]]

def evaluate(chunks, ranker, k=3, depth=10):
    rows = []
    for q, doc, ans in GOLD:
        rel = relevant_ids(doc, ans, chunks); ids = ranker(chunks, q, depth)
        rows.append((hit_at_k(ids, rel, k), reciprocal_rank(ids, rel, depth)))
    return {"hit@3": float(np.mean([r[0] for r in rows])), "MRR@10": float(np.mean([r[1] for r in rows]))}

results = pd.DataFrame({
    "hash 280/60":  evaluate(build(280, 60), rank_hash),
    "hash 120/0":   evaluate(build(120, 0), rank_hash),
    "tfidf 280/60": evaluate(build(280, 60), rank_tfidf),
}).T.round(3)
results

,hit@3,MRR@10
hash 280/60,1.00,1.00
hash 120/0,0.75,0.66
tfidf 280/60,1.00,1.00


In [12]:
def best_configurations( metrics ):
    best_hit = max(
        values["hit@3"]
        for values in metrics.values()
    )
    best_mrr = max(
        values["MRR@10"]
        for values in metrics.values()
        if values["hit@3"] == best_hit
    )
    best_names = []
    for name, values in metrics.items():
        if ( values["hit@3"] == best_hit and values["MRR@10"] == best_mrr ):
            best_names.append( name )
    return sorted( best_names )

In [13]:
_metrics = {"hash": {"hit@3": 0.75, "MRR@10": 0.66},
            "tfidf": {"hit@3": 1.0, "MRR@10": 0.9}}
assert best_configurations(_metrics) == ["tfidf"]